<h1 align="center" style="color: #2E5B88;">Text Preprocessing for Machine Learning Models</h1>

In [3]:
# Core libraries
import re
import string

# Data handling
import pandas as pd
import numpy as np
from langdetect import detect, DetectorFactory
import re
import html
import emoji
from bs4 import BeautifulSoup
# Text preprocessing
from nltk.corpus import stopwords
from lingua import LanguageDetectorBuilder
from nltk.stem import WordNetLemmatizer, PorterStemmer
from nltk.tokenize import word_tokenize, sent_tokenize
import nltk

nltk.download('stopwords')
nltk.download('punkt')
nltk.download('wordnet')


[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\jayas\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\jayas\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\jayas\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

# Loading Processed Dataset

This notebook begins by loading the cleaned dataset generated during the previous phase.

The dataset has already undergone:

- Data quality assessment
- Missing value handling
- Duplicate removal
- Feature creation
- Exploratory Data Analysis (EDA)

The objective of this notebook is to understand the textual characteristics of customer reviews, design an effective preprocessing pipeline, and generate numerical representations that can be used by machine learning and deep learning models.

In [46]:
df = pd.read_parquet("data/processed/reviews_after_eda.parquet")

In [3]:
df.head()

,rating,images,asin,parent_asin,user_id,timestamp,helpful_vote,verified_purchase,review,sentiment,char_length,word_count,sentence_count,year,month
0,5,[],B00YQ6X8EO,B00YQ6X8EO,AGKHLEW2SOWHNMFQIJGBECAF7INQ,2020-05-05 14:08:48.923,0,True,Such a lovely scent but not overpowering. This...,Positive,342,68,7,2020,5
1,4,[],B081TJ8YS3,B081TJ8YS3,AGKHLEW2SOWHNMFQIJGBECAF7INQ,2020-05-04 18:10:55.070,1,True,Works great but smells a little weird. This pr...,Positive,274,54,3,2020,5
2,5,[],B07PNNCSP9,B097R46CSY,AE74DYR3QUGVPZJ3P7RFWBGIX7XQ,2020-05-16 21:41:06.052,2,True,"Yes! Smells good, feels great!",Positive,30,5,2,2020,5
3,1,[],B09JS339BZ,B09JS339BZ,AFQLNQNQYFWQZPJQZS6V3NZU4QBQ,2022-01-28 18:13:50.220,0,True,Synthetic feeling Felt synthetic,Negative,32,4,1,2022,1
4,5,[],B08BZ63GMJ,B08BZ63GMJ,AFQLNQNQYFWQZPJQZS6V3NZU4QBQ,2020-12-30 10:02:43.534,0,True,A+ Love it,Positive,10,3,1,2020,12


In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 694195 entries, 0 to 694194
Data columns (total 15 columns):
 #   Column             Non-Null Count   Dtype         
---  ------             --------------   -----         
 0   rating             694195 non-null  int64         
 1   images             694195 non-null  object        
 2   asin               694195 non-null  object        
 3   parent_asin        694195 non-null  object        
 4   user_id            694195 non-null  object        
 5   timestamp          694195 non-null  datetime64[ns]
 6   helpful_vote       694195 non-null  int64         
 7   verified_purchase  694195 non-null  bool          
 8   review             694195 non-null  object        
 9   sentiment          694195 non-null  object        
 10  char_length        694195 non-null  int64         
 11  word_count         694195 non-null  int64         
 12  sentence_count     694195 non-null  int64         
 13  year               694195 non-null  int32   

In [5]:
# Define regex patterns
patterns = {
    "punctuations": r"[^\w\s]",              # any non-word, non-space character
    "emojis": r"[\U0001F600-\U0001F64F]",    # basic emoji range
    "numbers": r"\d+",                       # digits
    "html_code": r"<.*?>",                   # HTML tags
    "mentions": r"@\w+",                     # @mentions
    "emails": r"[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}" # email addresses
}

def count_inconsistencies(series):
    totals = {key: 0 for key in patterns.keys()}
    for text in series:
        text = str(text)
        for key, pattern in patterns.items():
            matches = re.findall(pattern, text)
            totals[key] += len(matches)
    return totals

# Apply to your DataFrame column "review"
totals = count_inconsistencies(df["review"])

# Print results
print("Text inconsistencies count across all reviews:")
for k, v in totals.items():
    print(f"{k}: {v}")


Text inconsistencies count across all reviews:
punctuations: 4459453
emojis: 11416
numbers: 274634
html_code: 149376
mentions: 297
emails: 46


In [7]:
# Initialize tools
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def preprocess_text(text):
    # 1. Lowercase
    text = text.lower()
    
    # 2. Tokenize
    tokens = nltk.word_tokenize(text)
    
    # 3. Remove stopwords and non-alphabetic tokens
    tokens = [word for word in tokens if word.isalpha() and word not in stop_words]
    
    # 4. Lemmatize
    tokens = [lemmatizer.lemmatize(word) for word in tokens]
    
    # Return as a single string
    return " ".join(tokens)

# Apply to your DataFrame column
df["clean_review"] = df["review"].apply(preprocess_text)

# Check a few samples
df[["review", "clean_review"]].head(10)


,review,clean_review
0,Such a lovely scent but not overpowering. This...,lovely scent overpowering spray really nice sm...
1,Works great but smells a little weird. This pr...,work great smell little weird product need wis...
2,"Yes! Smells good, feels great!",yes smell good feel great
3,Synthetic feeling Felt synthetic,synthetic feeling felt synthetic
4,A+ Love it,love
5,Pretty Color The polish was quiet thick and di...,pretty color polish quiet thick apply smoothly...
6,Handy Great for many tasks. I purchased these...,handy great many task purchased makeup removal...
7,Meh These were lightweight and soft but much t...,meh lightweight soft much small liking would p...
8,Great for at home use and so easy to use! This...,great home use easy use perfect salon visit us...
9,Nice shampoo for the money I get Keratin treat...,nice shampoo money get keratin treatment salon...


In [8]:
for i, review in enumerate(df["clean_review"].sample(10, random_state=42).tolist(), 1):
    print(f"{i}. {review}\n")


1. boo see difference comparing nyx hold leaf face little white washed really use anymore

2. camping must spending far inferior product finally found camping salt longer hard lump shaker sixpence well longer worry putting rice keep salt dry

3. terrible bag supper cheaply made fell apart easily would buy

4. beautiful got sister love beautiful got sister love

5. ingredient needed

6. era lo que esperaba malas los apliques se caen rápidamente

7. best best sunscreen ever used love everything

8. happened product used benefit brow zing kit last year wonderful ordered product changed worse wax shadow poor quality color product simply job prior change formulation brush attached made cheap quality box low quality well pricing poor quality product quite disappointment using product

9. great quality good bargain quality hair great hair last good proper maintenance love thickness real look appear look synthetic

10. sparkle little gem super shiney really like stone silver silver doesnt come

In [17]:
detector = LanguageDetectorBuilder.from_all_languages().build()

def lingua_detect(text):
    if not isinstance(text, str) or len(text.strip()) < 3:
        return "unknown"
    clean_text = text.replace("\n", " ").strip()
    detected_lang = detector.detect_language_of(clean_text)
    return detected_lang.iso_code_639_1.name.lower() if detected_lang else "unknown"

df["languages"] = df["review"].apply(lingua_detect)
df_english = df[df["languages"] == "en"].copy()

In [18]:
df["languages"].value_counts()

languages
en         674523
es          10976
la           1175
so           1080
sl            934
fr            617
pt            473
unknown       465
tl            465
zu            315
it            205
nl            203
af            196
nb            194
ro            163
ga            155
fi            147
da            140
nn            132
eo            113
cy            112
st            102
ca             99
xh             96
ts             95
sv             92
et             83
sw             81
sn             78
pl             75
de             70
bs             64
cs             62
yo             57
tn             50
sq             49
sk             49
lg             46
hu             42
eu             27
lt             18
id             17
ms             11
is             11
mi             10
tr              8
hr              8
vi              7
lv              2
ja              1
az              1
zh              1
Name: count, dtype: int64

In [38]:
df[df["languages"].isin(["es", "so","la","s1"])][["review", "languages"]].sample(20)


,review,languages
498074,"Lo que sea verdadera y util , Es a prueba de b...",es
29242,MASCARILLA PARA EL CABELLO EXCELNTE PRODUCTO,la
339256,Es bueno el producto Creía que eran de 3.0 onz...,es
459491,Good Good,so
513779,Hermoso regalo Tal como se muestra en las imág...,es
624445,No me gustó mucho Se descarga muy rápido y no ...,es
683336,Muy buena buena ponérsela por las noches Aclara,es
428172,Nice Nice,la
629081,Muy buen producto Muy buen producto me gusto,es
429440,Buena para principiantes No tiene tanta fuerza...,es


In [26]:
df[df["languages"] == "unknown"][["review", "languages"]].sample(10)

,review,languages
522760,👌😘 😊👌,unknown
109357,👍 🌟,unknown
3535,👍 👍,unknown
149931,👍 👍,unknown
180286,🙂 🙂,unknown
91914,👍👍👍👍👍 👍👍👍👍👍,unknown
232867,👍 👍,unknown
562584,👍👍👍👍👍👍 👍👌👍,unknown
211607,❤ ❤,unknown
519750,😀 👍👍,unknown


## Language Filtering

As part of the preprocessing pipeline, language detection was performed to ensure that the dataset was consistent with the objective of this project: **English sentiment analysis**.

Since the preprocessing techniques used in this project—such as English stopword removal, lemmatization, and feature extraction—are specifically designed for English text, reviews identified as non-English were removed from the dataset.

This filtering step reduces vocabulary noise, prevents multiple languages from being mixed within the same feature space, and improves the overall quality of the input data for machine learning and deep learning models.


In [21]:
total_before = len(df)
total_after = len(df_english)
removed_rows = total_before - total_after
percentage_retained = (total_after / total_before) * 100

print("========================================")
print("       DATASET FILTERING SUMMARY        ")
print("========================================")
print(f"Original Rows      : {total_before:,}")
print(f"English Rows Kept  : {total_after:,} ({percentage_retained:.2f}%)")
print(f"Foreign Rows Dropped: {removed_rows:,}")
print("========================================")


       DATASET FILTERING SUMMARY        
Original Rows      : 694,195
English Rows Kept  : 674,523 (97.17%)
Foreign Rows Dropped: 19,672


After language filtering, the final dataset contains **674,523 English customer reviews**, providing a large, linguistically consistent corpus for feature engineering and sentiment classification.

In [22]:
df_english.head()

,rating,images,asin,parent_asin,user_id,timestamp,helpful_vote,verified_purchase,review,sentiment,char_length,word_count,sentence_count,year,month,clean_review,languages
0,5,[],B00YQ6X8EO,B00YQ6X8EO,AGKHLEW2SOWHNMFQIJGBECAF7INQ,2020-05-05 14:08:48.923,0,True,Such a lovely scent but not overpowering. This...,Positive,342,68,7,2020,5,lovely scent overpowering spray really nice sm...,en
1,4,[],B081TJ8YS3,B081TJ8YS3,AGKHLEW2SOWHNMFQIJGBECAF7INQ,2020-05-04 18:10:55.070,1,True,Works great but smells a little weird. This pr...,Positive,274,54,3,2020,5,work great smell little weird product need wis...,en
2,5,[],B07PNNCSP9,B097R46CSY,AE74DYR3QUGVPZJ3P7RFWBGIX7XQ,2020-05-16 21:41:06.052,2,True,"Yes! Smells good, feels great!",Positive,30,5,2,2020,5,yes smell good feel great,en
3,1,[],B09JS339BZ,B09JS339BZ,AFQLNQNQYFWQZPJQZS6V3NZU4QBQ,2022-01-28 18:13:50.220,0,True,Synthetic feeling Felt synthetic,Negative,32,4,1,2022,1,synthetic feeling felt synthetic,en
5,4,[{'small_image_url': 'https://images-na.ssl-im...,B00R8DXL44,B00R8DXL44,AGMJ3EMDVL6OWBJF7CA5RGJLXN5A,2020-08-27 22:30:08.138,0,True,Pretty Color The polish was quiet thick and di...,Positive,138,26,2,2020,8,pretty color polish quiet thick apply smoothly...,en


In [23]:
# Function to count inconsistencies again
def count_inconsistencies(series):
    totals = {key: 0 for key in patterns.keys()}
    for text in series:
        text = str(text)
        for key, pattern in patterns.items():
            matches = re.findall(pattern, text)
            totals[key] += len(matches)
    return totals

# Count before and after
before_counts = count_inconsistencies(df_english["review"])
after_counts = count_inconsistencies(df_english["clean_review"])

print("Before cleaning:", before_counts)
print("After cleaning:", after_counts)

Before cleaning: {'punctuations': 4429496, 'emojis': 10504, 'numbers': 269837, 'html_code': 148744, 'mentions': 294, 'emails': 45}
After cleaning: {'punctuations': 0, 'emojis': 0, 'numbers': 0, 'html_code': 0, 'mentions': 0, 'emails': 0}


<h1 align="center" style="color: #2E5B88;">Light Text Preprocessing for Deep Learning Models</h1>

In [4]:
df_english.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 674523 entries, 0 to 674522
Data columns (total 17 columns):
 #   Column             Non-Null Count   Dtype         
---  ------             --------------   -----         
 0   rating             674523 non-null  int64         
 1   images             674523 non-null  object        
 2   asin               674523 non-null  object        
 3   parent_asin        674523 non-null  object        
 4   user_id            674523 non-null  object        
 5   timestamp          674523 non-null  datetime64[ns]
 6   helpful_vote       674523 non-null  int64         
 7   verified_purchase  674523 non-null  bool          
 8   review             674523 non-null  object        
 9   sentiment          674523 non-null  object        
 10  char_length        674523 non-null  int64         
 11  word_count         674523 non-null  int64         
 12  sentence_count     674523 non-null  int64         
 13  year               674523 non-null  int32   

In [5]:
def preprocess_text_dl(text):

    if not isinstance(text, str):
        return ""

    # Lowercase
    text = text.lower()

    # Decode HTML entities
    text = html.unescape(text)

    # Remove HTML
    text = BeautifulSoup(text, "html.parser").get_text(separator=" ")

    # Remove URLs
    text = re.sub(r'https?://\S+|www\.\S+', '', text)

    # Remove Emails
    text = re.sub(r'\S+@\S+', '', text)

    # Remove Mentions
    text = re.sub(r'@\w+', '', text)

    # Remove Emojis
    text = emoji.replace_emoji(text, replace="")

    # Normalize whitespace
    text = re.sub(r"\s+", " ", text).strip()

    return text

In [7]:
df_english["dl_review"] = df_english["review"].apply(preprocess_text_dl)

In [10]:
# Save dataset after all preprocessing
df_english.to_parquet(
    "data/processed/reviews_after_preprocessing.parquet",
    index=False
)

# Save Processed Dataset

The review dataset has now been preprocessed and standardized to support both classical machine learning and deep learning workflows.

Two text representations have been created:

- **`clean_review`** – An aggressively cleaned version of the review text designed for traditional machine learning models. This pipeline includes lowercasing, HTML removal, URL removal, email and mention removal, emoji removal, tokenization, stopword removal, filtering of non-alphabetic tokens, and lemmatization.

- **`dl_review`** – A lightly preprocessed version of the review text intended for deep learning models. This representation removes non-linguistic noise such as HTML tags, URLs, email addresses, mentions, emojis, and extra whitespace while preserving the original sentence structure, stopwords, punctuation, numbers, and word forms to retain semantic information.

The processed dataset is saved to disk to ensure reproducibility, maintain a consistent preprocessing pipeline across all experiments, and avoid repeating computationally expensive preprocessing steps in subsequent notebooks.

The next phase of the project focuses on **Feature Engineering**, where these text representations will be transformed into model-ready numerical formats. The **`clean_review`** column will be used to generate representations such as Bag of Words (BoW), TF-IDF, N-Grams, and other feature engineering techniques for classical machine learning models, while the **`dl_review`** column will be prepared through tokenization, sequence generation, and padding for deep learning architectures such as LSTM, Bidirectional LSTM, and GRU.